# Frequency-10 versus history-conditioned sample-5

This notebook compares three inference strategies on the original strict-extractive Qwen2.5-0.5B SFT adapter:

1. **Independent sample-10 + frequency top-5** — frozen baseline; MRR previously measured as 0.478125.
2. **Independent sample-5 in natural order** — controls for the smaller generation budget.
3. **History-conditioned sample-5 in natural order** — turn 1 reuses frozen independent sample 1; turns 2–5 show all previously found valid candidates and request a different evidence-supported expression.

The conditioned method does not alter or train the model. All methods use temperature 0.7, top-p 0.9, the same 4,096-token context, punctuation-preserving deduplication, and official BioASQ Java scoring.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
PYTHON = Path("/home/dinh-tuan/miniconda3/envs/bioasq/bin/python")
SCRIPT = ROOT / "cse_dpo/compare_frequency_vs_conditioned_sampling.py"
OUTPUT = ROOT / "Artifacts/cse_dpo/inference_strategy_comparisons/qwen25_05b_sft_frequency10_vs_conditioned5_t07_seed3407"
ENV = {**os.environ, "PYTHONPATH": str(ROOT)}

print("Script:", SCRIPT)
print("Output:", OUTPUT)


## Optional three-question smoke run

This creates a separate smoke3 output directory and does not affect the full run.


In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    subprocess.run(
        [str(PYTHON), str(SCRIPT), "--limit", "3"],
        cwd=ROOT,
        env=ENV,
        check=True,
    )
else:
    print("Set RUN_SMOKE=True to run the three-question GPU smoke test.")


## Full 160-question comparison

The script appends one completed question at a time, so rerunning this cell resumes safely.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    subprocess.run(
        [str(PYTHON), str(SCRIPT)],
        cwd=ROOT,
        env=ENV,
        check=True,
    )
else:
    print("Set RUN_FULL=True to start or resume the full comparison.")


## Review results


In [ ]:
summary_path = OUTPUT / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(summary["summaries"])
    display(summary["conditioned5_vs_frequency10"])
else:
    print("The full run has not finished yet:", summary_path)

comparison_path = OUTPUT / "question_level_comparison.csv"
if comparison_path.exists():
    import pandas as pd
    comparison = pd.read_csv(comparison_path)
    display(comparison.sort_values("conditioned5_minus_frequency10_mrr", ascending=False).head(10))
    display(comparison.sort_values("conditioned5_minus_frequency10_mrr").head(10))
